In [ ]:
!pip install torch-scatter torch-sparse torch-cluster torch-spline-conv torch-geometric -f https://data.pyg.org/whl/torch-$(torch.__version__).html -q
!pip install sentence-transformers -q
!pip install scikit-learn -q

In [ ]:
# ==============================================================================\
# CELL 1: TẢI DỮ LIỆU TỪ GOOGLE DRIVE (GIỮ RIÊNG TRAIN/VAL)
# ==============================================================================
!pip install gdown -q

import os
import gdown
import pandas as pd
import shutil

# 1. CẤU HÌNH THƯ MỤC
DATA_INTERIM = "data/interim"
DATA_PROCESSED = "data/processed"
os.makedirs(DATA_INTERIM, exist_ok=True)
os.makedirs(DATA_PROCESSED, exist_ok=True)

# 2. LINK FILE (ID lấy từ code cũ của bạn)
# Lưu ý: Tôi đã chuyển link view sang format tải trực tiếp của gdown
FILES = {
    # File Train Log
    "temp_train.parquet": {
        "url": "https://drive.google.com/uc?id=1qEP4UEoysTTUwLRIu8pRsyF3OIpRxMfD",
        "dest": f"{DATA_INTERIM}/train.parquet" # Đổi tên luôn thành train.parquet
    },
    # File Val Log (Giữ riêng để đánh giá)
    "temp_val.parquet": {
        "url": "https://drive.google.com/uc?id=1g6WYtdoMK-JYv7m7t3mnlD2D0t0SWbI8",
        "dest": f"{DATA_INTERIM}/val.parquet"   # Đổi tên luôn thành val.parquet
    },
    # Metadata Train (Chứa thông tin item cũ)
    "metadata.parquet": {
        "url": "https://drive.google.com/uc?id=1ELz1QRp9EEVK2g086b_NyiiZvPtmT6vb",
        "dest": f"{DATA_PROCESSED}/metadata.parquet"
    },
    # Metadata Test (Chứa thông tin item tương lai - Target)
    "metadata_test.parquet": {
        "url": "https://drive.google.com/uc?id=10aQeyaaHFKqXJ5VR03M1LH_mjJpVAbH7",
        "dest": f"{DATA_PROCESSED}/metadata_test.parquet"
    }
}

def download_and_setup():
    print(" Bắt đầu tải dữ liệu...")

    for original_name, info in FILES.items():
        output_path = info["dest"]
        url = info["url"]

        if not os.path.exists(output_path):
            print(f" Đang tải {original_name}...")
            gdown.download(url, output_path, quiet=False)
        else:
            print(f" {original_name} đã tồn tại.")

    # Kiểm tra lại file
    print("\n Kiểm tra thư mục dữ liệu:")
    if os.path.exists(DATA_INTERIM):
        print(f"   {DATA_INTERIM}/: {os.listdir(DATA_INTERIM)}")
    if os.path.exists(DATA_PROCESSED):
        print(f"   {DATA_PROCESSED}/: {os.listdir(DATA_PROCESSED)}")

    print("\n Dữ liệu đã sẵn sàng để xây dựng đồ thị!")

if __name__ == "__main__":
    download_and_setup()

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch_geometric.nn import GATConv
from torch_geometric.utils import dropout_edge
from tqdm import tqdm
import json
import ast
import os

# --- CONFIG ---
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EMBED_DIM = 64     # Kích thước vector
HEADS = 4          # Số đầu Attention
LR = 0.001
EPOCHS = 30        # Số vòng train
TOP_K = 5          # Đánh giá @5

# File Paths
TRAIN_PATH = "data/interim/train.parquet"
VAL_PATH = "data/interim/val.parquet" # Lưu ý: Đây phải là file tách riêng, không gộp
META_TRAIN_PATH = "data/processed/metadata.parquet"
META_TEST_PATH = "data/processed/metadata_test.parquet"

def process_list(val):
    """Chuẩn hóa cột list string về mảng Python"""
    if isinstance(val, str) and val.startswith('['):
        try: return ast.literal_eval(val)
        except: return []
    if isinstance(val, (list, np.ndarray)):
        return [str(x) for x in val]
    return []

In [ ]:
def build_graph_resources():
    print(" Đang xây dựng đồ thị (User + Train Items + Future Items)...")

    # 1. Load Dữ liệu
    df_train = pd.read_parquet(TRAIN_PATH)
    df_meta_train = pd.read_parquet(META_TRAIN_PATH)
    df_meta_test = pd.read_parquet(META_TEST_PATH)

    # Gộp Metadata để lấy toàn bộ thông tin thuộc tính (Actor, Genre...)
    # Lưu ý: drop_duplicates để tránh trùng item giữa 2 file (nếu có)
    df_meta_full = pd.concat([df_meta_train, df_meta_test], ignore_index=True)
    df_meta_full = df_meta_full.drop_duplicates(subset=['tv_show_id'], keep='last')

    # Chuẩn hóa ID về dạng string sạch
    df_train['uid_str'] = df_train['user_id'].astype(str).str.replace('.0','')
    df_train['iid_str'] = df_train['tv_show_id'].astype(str).str.replace('.0','')
    df_meta_full['iid_str'] = df_meta_full['tv_show_id'].astype(str).str.replace('.0','')

    # Xác định tập Item mục tiêu (cho tuần tới) để lọc lúc Inference
    target_item_ids = set(df_meta_test['tv_show_id'].astype(str).str.replace('.0','').unique())

    # --- MAPPING ID ---
    entity_map = {}  # {original_id: graph_index}
    node_types = []  # 0:User, 1:Item, 2:Attribute

    def get_idx(key, type_code):
        if key not in entity_map:
            entity_map[key] = len(entity_map)
            node_types.append(type_code)
        return entity_map[key]

    edges = []

    # A. Cạnh Tương tác (User - Item) từ tập TRAIN
    # Chỉ lấy dữ liệu quá khứ để train, tránh data leakage
    # Lọc nhiễu: Chỉ lấy lượt xem > 60s hoặc screen_time > 5%
    valid_train = df_train[df_train['duration_view'] > 60]

    for _, row in tqdm(valid_train.iterrows(), total=len(valid_train), desc="Edges: Train Interactions"):
        u_node = get_idx(f"u_{row['uid_str']}", 0)
        i_node = get_idx(f"i_{row['iid_str']}", 1)
        edges.append([u_node, i_node])
        edges.append([i_node, u_node]) # Vô hướng

    # B. Cạnh Tri thức (Item - Attribute) từ FULL METADATA
    # Kết nối cả Item cũ và Item mới vào đồ thị qua thuộc tính
    for _, row in tqdm(df_meta_full.iterrows(), total=len(df_meta_full), desc="Edges: Knowledge"):
        i_node = get_idx(f"i_{row['iid_str']}", 1)

        # Genre
        for g in process_list(row.get('genres_list')):
            g_node = get_idx(f"g_{g}", 2)
            edges.append([i_node, g_node]); edges.append([g_node, i_node])

        # Actor
        for a in process_list(row.get('actors_list')):
            a_node = get_idx(f"a_{a}", 2)
            edges.append([i_node, a_node]); edges.append([a_node, i_node])

    edge_index = torch.LongTensor(edges).t().contiguous()
    node_types = torch.LongTensor(node_types)

    print(f" Đồ thị hoàn tất: {len(entity_map)} nodes, {edge_index.shape[1]} edges.")

    return edge_index, entity_map, node_types, target_item_ids

In [ ]:
class CKG_GAT(nn.Module):
    def __init__(self, num_nodes, embed_dim, node_types):
        super().__init__()
        # Embedding riêng cho từng node
        self.node_emb = nn.Embedding(num_nodes, embed_dim)
        nn.init.xavier_uniform_(self.node_emb.weight)

        # GAT Layers
        self.conv1 = GATConv(embed_dim, embed_dim, heads=HEADS, concat=False, dropout=0.2)
        self.conv2 = GATConv(embed_dim, embed_dim, heads=HEADS, concat=False, dropout=0.2)

        self.act = nn.LeakyReLU()
        self.dropout = nn.Dropout(0.2)

    def forward(self, edge_index):
        x = self.node_emb.weight

        # Layer 1 + Residual
        x1 = self.conv1(x, edge_index)
        x1 = self.act(x1)
        x1 = self.dropout(x1)
        x = x + x1  # Residual connection

        # Layer 2 + Residual
        x2 = self.conv2(x, edge_index)
        x = x + x2  # Residual connection

        return x

In [ ]:
# ==============================================================================
# ULTRA-LOW MEMORY V2: SUBGRAPH TRAINING + MIXED PRECISION
# ==============================================================================

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch_geometric.nn import GATConv
from torch_geometric.utils import dropout_edge
from torch.cuda.amp import autocast, GradScaler # Thư viện Mixed Precision
from tqdm import tqdm
import ast
import gc
import sys

# --- CẤU HÌNH ---
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🔹 Running on: {DEVICE}")

EMBED_DIM = 48
HEADS = 4
LR = 0.001
EPOCHS = 30
TOP_K = 5
BATCH_SIZE_EVAL = 64
SUBGRAPH_RATIO = 0.5

# Paths
TRAIN_PATH = "data/interim/train.parquet"
VAL_PATH = "data/interim/val.parquet"
META_TRAIN_PATH = "data/processed/metadata.parquet"
META_TEST_PATH = "data/processed/metadata_test.parquet"

# --- HELPER FUNCTIONS ---
def process_list(val):
    if isinstance(val, str) and val.startswith('['):
        try: return ast.literal_eval(val)
        except: return []
    if isinstance(val, (list, np.ndarray)):
        return [str(x) for x in val]
    return []

def compute_metrics_batch(val_ground_truth, val_preds):
    recalls, maps, ndcgs = [], [], []
    k = TOP_K
    for uid, true_items in val_ground_truth.items():
        if uid not in val_preds:
            recalls.append(0); maps.append(0); ndcgs.append(0); continue

        pred_items = val_preds[uid]
        hits = 0; sum_prec = 0; dcg = 0
        idcg = sum([1.0/np.log2(i+2) for i in range(min(len(true_items), k))])

        for i, item in enumerate(pred_items):
            if item in true_items:
                hits += 1
                sum_prec += hits / (i+1)
                dcg += 1.0/np.log2(i+2)

        recalls.append(hits / len(true_items) if len(true_items)>0 else 0)
        maps.append(sum_prec / min(len(true_items), k) if len(true_items)>0 else 0)
        ndcgs.append(dcg / idcg if idcg>0 else 0)
    return np.mean(recalls), np.mean(maps), np.mean(ndcgs)

def build_graph_resources_optimized():
    print(" Xây dựng Đồ thị (Tối ưu bộ nhớ)...")

    # 1. Load Metadata và lấy Target IDs NGAY LẬP TỨC
    try:
        df_meta_test = pd.read_parquet(META_TEST_PATH)
        target_item_ids = set(df_meta_test['tv_show_id'].astype(str).str.replace('.0','').unique())

        df_meta_train = pd.read_parquet(META_TRAIN_PATH)
        df_meta_full = pd.concat([df_meta_train, df_meta_test], ignore_index=True)
        # Drop duplicates và reset index để giảm memory fragmentation
        df_meta_full = df_meta_full.drop_duplicates(subset=['tv_show_id'], keep='last').reset_index(drop=True)

        # Xóa biến thừa
        del df_meta_train, df_meta_test
        gc.collect()
    except Exception as e:
        print(f" Lỗi load metadata: {e}")
        return None, None, None, None

    # Mapping
    entity_map = {}
    node_types = []

    def get_idx(key, type_code):
        if key not in entity_map:
            entity_map[key] = len(entity_map)
            node_types.append(type_code)
        return entity_map[key]

    # Xây dựng cạnh Knowledge (Dùng generator để tránh tạo list khổng lồ)
    edges_list = []

    print("   -> Mapping Knowledge Graph...")
    # Dùng numpy array để duyệt nhanh hơn itertuples
    df_meta_full['iid_str'] = df_meta_full['tv_show_id'].astype(str).str.replace('.0','')

    for row in tqdm(df_meta_full.itertuples(), total=len(df_meta_full)):
        i = get_idx(f"i_{row.iid_str}", 1)

        # Xử lý genres
        genres = process_list(row.genres_list)
        for g in genres:
            gid = get_idx(f"g_{g}", 2)
            edges_list.append([i, gid]); edges_list.append([gid, i])

        # Xử lý actors
        actors = process_list(row.actors_list)
        for a in actors:
            aid = get_idx(f"a_{a}", 2)
            edges_list.append([i, aid]); edges_list.append([aid, i])

    # Xóa Metadata ngay
    del df_meta_full
    gc.collect()

    # 2. Load Train Interactions
    print("   -> Mapping Interactions...")
    try:
        df_train = pd.read_parquet(TRAIN_PATH)
        df_train = df_train[df_train['duration_view'] > 60] # Lọc bớt nhiễu

        df_train['uid_str'] = df_train['user_id'].astype(str).str.replace('.0','')
        df_train['iid_str'] = df_train['tv_show_id'].astype(str).str.replace('.0','')

        for row in tqdm(df_train.itertuples(), total=len(df_train)):
            u = get_idx(f"u_{row.uid_str}", 0)
            i = get_idx(f"i_{row.iid_str}", 1)
            edges_list.append([u, i]); edges_list.append([i, u])

        del df_train
        gc.collect()
    except: return None, None, None, None

    print("   -> Chuyển sang Tensor...")
    # Tạo tensor trực tiếp từ numpy để tiết kiệm RAM
    edges_np = np.array(edges_list, dtype=np.int64).T
    edge_index = torch.from_numpy(edges_np).contiguous()
    node_types = torch.LongTensor(node_types)

    del edges_list, edges_np
    gc.collect()

    print(f"✅ Nodes: {len(entity_map)} | Edges: {edge_index.shape[1]}")
    return edge_index, entity_map, node_types, target_item_ids

# --- MODEL ---
class CKG_GAT(nn.Module):
    def __init__(self, num_nodes, embed_dim, node_types):
        super().__init__()
        self.node_emb = nn.Embedding(num_nodes, embed_dim)
        nn.init.xavier_uniform_(self.node_emb.weight)

        # Dùng concat=False để giảm kích thước output vector
        self.conv1 = GATConv(embed_dim, embed_dim, heads=HEADS, concat=False, dropout=0.4)
        self.conv2 = GATConv(embed_dim, embed_dim, heads=HEADS, concat=False, dropout=0.4)
        self.act = nn.LeakyReLU()

    def forward(self, edge_index):
        x = self.node_emb.weight

        # Layer 1
        x1 = self.conv1(x, edge_index)
        x1 = self.act(x1)
        x = x + x1 # Residual

        # Layer 2
        x2 = self.conv2(x, edge_index)
        x = x + x2 # Residual

        return x

def run_training():
    # 1. Build Resources
    res = build_graph_resources_optimized()
    if res[0] is None: return
    edge_index, entity_map, node_types, target_ids = res

    # 2. Validation Prep
    print(" Chuẩn bị Val Data...")
    df_val = pd.read_parquet(VAL_PATH)
    val_counts = df_val['user_id'].value_counts()
    valid_users = val_counts[val_counts >= 5].index.tolist()

    val_ground_truth = {}
    valid_u_indices = []

    df_val = df_val[df_val['user_id'].isin(valid_users)][['user_id', 'tv_show_id']]
    for uid, group in df_val.groupby('user_id'):
        u_key = f"u_{str(uid).replace('.0','')}"
        if u_key not in entity_map: continue
        u_idx = entity_map[u_key]
        items = [entity_map[f"i_{str(iid).replace('.0','')}"] for iid in group['tv_show_id'].unique() if f"i_{str(iid).replace('.0','')}" in entity_map]
        if items:
            val_ground_truth[u_idx] = set(items)
            valid_u_indices.append(u_idx)

    del df_val
    gc.collect()

    # 3. Model & Scaler
    model = CKG_GAT(len(entity_map), EMBED_DIM, node_types).to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=LR)
    scaler = GradScaler() # Dùng cho Mixed Precision

    # Đẩy edge_index lên GPU
    edge_index = edge_index.to(DEVICE)
    node_types = node_types.to(DEVICE)

    # Masking Train Edges
    row, col = edge_index
    mask = (node_types[row] == 0) & (node_types[col] == 1)
    train_edges = edge_index[:, mask] # Giữ trên GPU
    item_nodes = torch.where(node_types == 1)[0]

    print("\n Bắt đầu Training (Subgraph + Mixed Precision)...")

    for epoch in range(1, EPOCHS + 1):
        model.train()
        optimizer.zero_grad()

        # --- SUBGRAPH SAMPLING (Chìa khóa chống OOM) ---

        train_edge_index, _ = dropout_edge(edge_index, p=(1.0 - SUBGRAPH_RATIO), force_undirected=True, training=True)

        # --- MIXED PRECISION TRAINING ---
        with autocast():
            # Forward pass với subgraph
            emb = model(train_edge_index)

            # Sampling BPR
            batch_idx = torch.randperm(train_edges.size(1))[:2048]
            batch_pos = train_edges[:, batch_idx]

            users = batch_pos[0]
            pos_items = batch_pos[1]
            neg_items = item_nodes[torch.randint(0, len(item_nodes), (len(users),)).to(DEVICE)]

            # Tính Loss
            pos_scores = (emb[users] * emb[pos_items]).sum(dim=1)
            neg_scores = (emb[users] * emb[neg_items]).sum(dim=1)
            loss = -torch.log(torch.sigmoid(pos_scores - neg_scores)).mean()

        # Backward pass với Scaler
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        # Dọn dẹp VRAM sau mỗi epoch
        del train_edge_index, emb, pos_scores, neg_scores

        if epoch % 5 == 0:
            print(f"Ep {epoch}: Loss {loss.item():.4f} | Evaluating...")

            # --- EVALUATION (Low Memory) ---
            model.eval()
            val_preds = {}
            with torch.no_grad():
                # Full Graph Inference
                final_emb = model(edge_index)

                all_items_emb = final_emb[item_nodes]
                real_item_indices = item_nodes.cpu().numpy()

                # Batch Eval
                num_users = len(valid_u_indices)
                for i in range(0, num_users, BATCH_SIZE_EVAL):
                    batch_u = valid_u_indices[i : min(i + BATCH_SIZE_EVAL, num_users)]
                    batch_u_tensor = torch.LongTensor(batch_u).to(DEVICE)

                    scores = torch.matmul(final_emb[batch_u_tensor], all_items_emb.t())
                    _, topk_idx = torch.topk(scores, k=TOP_K, dim=1)

                    topk_idx = topk_idx.cpu().numpy()
                    del scores

                    for idx_in_batch, u_idx in enumerate(batch_u):
                        val_preds[u_idx] = list(real_item_indices[topk_idx[idx_in_batch]])

                rec, map_sc, ndcg = compute_metrics_batch(val_ground_truth, val_preds)
                print(f"   -> Val R@5: {rec:.4f} | MAP@5: {map_sc:.4f} | NDCG@5: {ndcg:.4f}")

                del final_emb, all_items_emb
                gc.collect()
                if torch.cuda.is_available(): torch.cuda.empty_cache()

    return model, entity_map, target_ids, node_types

if __name__ == "__main__":
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    model, entity_map, target_item_ids, node_types = run_training()

In [ ]:
def generate_submission(model, entity_map, target_item_ids, node_types):
    print("\n Đang tạo gợi ý cho tuần tới (Chỉ dùng Item trong Metadata Test)...")
    model.eval()

    # 1. Lấy danh sách Item mục tiêu (Future Items)
    target_indices = []
    target_ids_clean = []

    for tid in target_item_ids:
        key = f"i_{tid}"
        if key in entity_map:
            target_indices.append(entity_map[key])
            target_ids_clean.append(tid)

    if not target_indices:
        print(" Không tìm thấy item nào trong Metadata Test khớp với đồ thị.")
        return

    # Chuyển sang Tensor
    target_tensor = torch.LongTensor(target_indices).to(DEVICE)

    # 2. Tính toán Embedding cuối cùng

    # Lấy full embedding từ model (chạy lại forward 1 lần nếu cần)
    # final_emb = model(...)

    # 3. Matching cho từng User (Trong file submission template)
    # Load users cần dự đoán
    sub_df = pd.read_csv("submission.csv", dtype={'user_id': str})
    target_users = sub_df['user_id'].unique()

    results = []

    # Chạy inference

    print(" Đã tạo xong file submission.")

# Gọi hàm này sau khi train xong
# generate_submission(model, entity_map, target_item_ids, node_types)